RMRB Analysis v1
- (2024.9.11) Change text extraction modular from PyPDF2 to pdfplumder. The reasons are as following: 
  1. When it comes to ciphertext, `pdfplumder` can interpret elements as ASCII chars. But for `PyPDF2`, elements will be interpreted as original symbols (like `!!"!`), making it less convenient to recognize ciphertext.
  2. Although `PyPDF2` can parse the plaintext more formatted and `pdfplumder` is in mess, both `pdfplumder` and `PyPDF2` can find string "广告" correctly.

In [11]:
from PyPDF2 import PdfReader, PdfWriter
import pdfplumber
from pdf2image import convert_from_path
import pytesseract
from datetime import datetime, timedelta
from pathlib import Path
import os

from pdfminer.pdfdocument import PDFDocument
from pdfminer.pdfparser import PDFParser
from pdfminer.pdfinterp import PDFResourceManager, PDFPageInterpreter
from pdfminer.converter import PDFPageAggregator
from pdfminer.layout import LTTextBoxHorizontal, LAParams
from pdfminer.high_level import extract_text
# from pdfminer.pdfinterp import PDFTextExtractionNotAllowed

# Set the path to the Tesseract installation
pytesseract.pytesseract.tesseract_cmd = "D:\\Tesseract-OCR\\tesseract.exe"

In [12]:
Advertisement = "广告"
Cipher_AD = """!!"!"""

In [13]:
ROOT_PATH = "D:/AI_data_analysis/RMRB/"
RMRB2024090616 = "D:/AI_data_analysis/RMRB/2024090616.pdf"
def PATH_FUN(YEAR, MONTH, DAY, EDITION, SUFFIX, NAME_bool=False):
    Split_Year = ["2023"] 
    NAME = YEAR + MONTH + DAY
    INFO_PATH = (
        NAME 
        + (
            ("/" + NAME + EDITION) 
            if YEAR in Split_Year else ""
        ))
    SUFFIX = ".pdf"
    pdf_path = ROOT_PATH + YEAR + "/" + INFO_PATH + SUFFIX
    if NAME_bool:
        return pdf_path, NAME
    else: return  pdf_path
PDF_PATH = PATH_FUN("2022", "01", "02", "02", ".pdf")
PDF_PATH

'D:/AI_data_analysis/RMRB/2022/20220102.pdf'

In [14]:
"""
D:/AI_data_analysis/RMRB/2018/20180216.pdf
D:/AI_data_analysis/RMRB/2018/20180425.pdf
D:/AI_data_analysis/RMRB/2018/20180516.pdf
D:/AI_data_analysis/RMRB/2018/20180726.pdf
D:/AI_data_analysis/RMRB/2018/20180817.pdf
D:/AI_data_analysis/RMRB/2018/20180818.pdf
D:/AI_data_analysis/RMRB/2018/20181012.pdf

D:/AI_data_analysis/RMRB/2019/20190107.pdf
D:/AI_data_analysis/RMRB/2019/20190402.pdf
D:/AI_data_analysis/RMRB/2019/20190410.pdf
D:/AI_data_analysis/RMRB/2019/20190412.pdf
D:/AI_data_analysis/RMRB/2019/20190728.pdf
D:/AI_data_analysis/RMRB/2019/20190802.pdf

"""
def Generate_Dates(year):
    start_date = datetime(year, 1, 1)
    end_date = datetime(year + 1, 1, 1)
    current_date = start_date
    
    dates = []
    while current_date < end_date:
        dates.append(current_date.strftime('%Y%m%d'))
        current_date += timedelta(days=1)
    return dates

def Check_PDF_Exist(YEAR):
    MISSING_PDF = []
    start_date = datetime(int(YEAR), 1, 1)
    end_date = datetime(int(YEAR) + 1, 1, 1)
    current_date = start_date
    while current_date < end_date:
        MONTH = str(current_date.month)
        MONTH = ("0" + MONTH) if len(MONTH) == 1 else MONTH
        DAY = str(current_date.day)
        DAY = ("0" + DAY) if len(DAY) == 1 else DAY
        PATH = PATH_FUN(YEAR, MONTH, DAY, "", ".pdf")
        try:
            with open(PATH, "rb") as file:
                PdfReader(file)
        except FileNotFoundError: 
            print(PATH)
            MISSING_PDF.append(PATH)
        current_date += timedelta(days=1)
        return MISSING_PDF
# Check_PDF_Exist("2018")

In [15]:
def Count_Files(folder_path):
    # Create a Path object for the folder
    folder = Path(folder_path)
    # Count all files in the folder
    file_count = sum(1 for file in folder.iterdir() if file.is_file())
    return file_count
# Count_Files("D:/AI_data_analysis/RMRB")

def Check_Folder_Exist(folder_path):
    # Check if the folder exists
    if not os.path.exists(folder_path):
        # Create the folder if it does not exist
        os.makedirs(folder_path)
        print(f'Folder created: {folder_path}')
    else:
        print(f'Folder already exists: {folder_path}')
# Check_Folder_Exist("D:/AI_data_analysis/RMRB")

In [16]:
# Generate image for all ad pages using OCR
def Genetare_AD_Image(YEAR):
    FOLD_PATH = ROOT_PATH + f"{YEAR}_AD"
    Check_Folder_Exist(FOLD_PATH)
    start_date = datetime(int(YEAR), 1, 1)
    end_date = datetime(int(YEAR) + 1, 1, 1)
    current_date = start_date
    while current_date < end_date:
        MONTH = str(current_date.month)
        MONTH = ("0" + MONTH) if len(MONTH) == 1 else MONTH
        DAY = str(current_date.day)
        DAY = ("0" + DAY) if len(DAY) == 1 else DAY
        PDF_PATH, NAME = PATH_FUN(YEAR, MONTH, DAY, "", ".pdf", NAME_bool=True)
        print("PDF PATH:", PDF_PATH)
        with pdfplumber.open(PDF_PATH) as PDF:
            # PDF = PdfReader(file)
            PAGES_NUM = len(PDF.pages)
            for page_num in range(PAGES_NUM):
                text = PDF.pages[page_num].extract_text().replace(" ", "")
                IMAGE_PATH = FOLD_PATH + "/" + f"{NAME}_{page_num+1}_AD.png"
                if Advertisement in text: # First filter: plaintext "广告"
                    # Attension: first_page and last_page starts at 1
                    IMAGE = convert_from_path(PDF_PATH, first_page=page_num+1, last_page=page_num+1)
                    IMAGE[0].save(IMAGE_PATH, "PNG") # Only one image in `IMAGE`
                    print("Full-page Ad:", IMAGE_PATH)
                elif """!!"!""" in text: # Second filter: ciphertext "广告"
                    # Attension: first_page and last_page starts at 1
                    IMAGE = convert_from_path(PDF_PATH, first_page=page_num+1, last_page=page_num+1)
                    IMAGE[0].save(IMAGE_PATH, "PNG") # Only one image in `IMAGE`
                    print("Half-page Ad", IMAGE_PATH)
        current_date += timedelta(days=1)
# Genetare_AD_Image("2022")

In [17]:
# 89 mins
# Only Second filter
# Genetare_AD_Image("2021")

In [18]:
"""
Exception: 2020011712
2020040712
2020040715
"""
# Genetare_AD_Image("2020")

'\nException: 2020011712\n2020040712\n2020040715\n'

In [19]:
Genetare_AD_Image("2019")

Folder created: D:/AI_data_analysis/RMRB/2019_AD
PDF PATH: D:/AI_data_analysis/RMRB/2019/20190101.pdf
Full-page Ad: D:/AI_data_analysis/RMRB/2019_AD/20190101_8_AD.png
PDF PATH: D:/AI_data_analysis/RMRB/2019/20190102.pdf
Full-page Ad: D:/AI_data_analysis/RMRB/2019_AD/20190102_8_AD.png
Full-page Ad: D:/AI_data_analysis/RMRB/2019_AD/20190102_12_AD.png
Full-page Ad: D:/AI_data_analysis/RMRB/2019_AD/20190102_15_AD.png
Full-page Ad: D:/AI_data_analysis/RMRB/2019_AD/20190102_20_AD.png
PDF PATH: D:/AI_data_analysis/RMRB/2019/20190103.pdf
Full-page Ad: D:/AI_data_analysis/RMRB/2019_AD/20190103_8_AD.png
Half-page Ad D:/AI_data_analysis/RMRB/2019_AD/20190103_12_AD.png
Full-page Ad: D:/AI_data_analysis/RMRB/2019_AD/20190103_20_AD.png
PDF PATH: D:/AI_data_analysis/RMRB/2019/20190104.pdf
Full-page Ad: D:/AI_data_analysis/RMRB/2019_AD/20190104_6_AD.png
Full-page Ad: D:/AI_data_analysis/RMRB/2019_AD/20190104_8_AD.png
Full-page Ad: D:/AI_data_analysis/RMRB/2019_AD/20190104_20_AD.png
PDF PATH: D:/AI_dat